In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import os
import requests

# Define the dataset file name and URL
file_name = "winequality-red.csv"
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"

# Check if the file exists, if not, download it
if not os.path.exists(file_name):
    print(f"Downloading {file_name}...")
    response = requests.get(url)
    response.raise_for_status() # Raise an exception for bad status codes
    with open(file_name, "w") as f:
        f.write(response.text)
    print(f"{file_name} downloaded successfully.")
else:
    print(f"{file_name} already exists.")

# Load dataset (note: this dataset uses semicolons as separators)
df = pd.read_csv(file_name, sep=';')

# Binary classification target
df['label'] = (df['quality'] >= 6).astype(int)

X = df.drop(['quality','label'], axis=1)
y = df['label']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)



models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, random_state=42),
    "kNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1] if hasattr(model,"predict_proba") else y_pred

    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred)
    }
    results.append(metrics)

results_df = pd.DataFrame(results)
print(results_df)




!pip install streamlit
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# The models have already been trained and are available in the 'models' dictionary above.
# The following lines were causing a FileNotFoundError because the model files were not saved.
# If you intend to save and load models, you would need to add joblib.dump() calls after training.
# import joblib
# models = {
#     "Logistic Regression": joblib.load("model/logreg.pkl"),
#     "Decision Tree": joblib.load("model/dtree.pkl")
# }


with open('app.py', 'w') as f:
    f.write('''
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

st.title('Wine Quality Prediction Dashboard')

# Load the data and models from the Colab environment
# In a real-world scenario, these would be loaded from saved files or a database
# For this demonstration, we assume X_test, y_test, models, and results_df are available from the Colab notebook's global scope

# To make this app runnable independently, we'd need to save/load these objects.
# For now, we'll assume the script is run within the Colab environment after the training cell.

# Placeholder for demonstration if running directly outside Colab after saving state
# In a complete deployment, you'd load models and data from persistent storage.

# Example of how to get data if this were run standalone (requires saving X_test, y_test, etc.):
# try:
#     X_test = pd.read_csv('X_test.csv')
#     y_test = pd.read_csv('y_test.csv')['label']
#     # Load models from .pkl files if they were saved
#     import joblib
#     models = {
#         "Logistic Regression": joblib.load("model/logreg.pkl"),
#         "Decision Tree": joblib.load("model/dtree.pkl"),
#         # ... load other models
#     }
#     results_df = pd.read_csv('results_df.csv')
# except FileNotFoundError:
#     st.error("Please run the model training cell in Colab first to generate data and models, or ensure necessary files are saved.")
#     st.stop()

# Display overall performance
st.header("Overall Model Performance")
st.dataframe(results_df)

# Model selection
selected_model_name = st.selectbox('Select a model to view detailed analysis:', list(models.keys()))

if selected_model_name:
    selected_model = models[selected_model_name]

    st.subheader(f'Detailed Analysis for {selected_model_name}')

    # Make predictions
    y_pred = selected_model.predict(X_test)

    # Classification Report
    st.write("### Classification Report")
    report = classification_report(y_test, y_pred, output_dict=True)
    st.json(report)

    # Confusion Matrix
    st.write("### Confusion Matrix")
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots()
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'Confusion Matrix for {selected_model_name}')
    st.pyplot(fig)

''')
print("app.py created successfully!")



winequality-red.csv already exists.
                 Model  Accuracy       AUC  Precision    Recall        F1  \
0  Logistic Regression  0.740625  0.819050   0.785714  0.737430  0.760807   
1        Decision Tree  0.715625  0.710131   0.765060  0.709497  0.736232   
2                  kNN  0.706250  0.773743   0.720207  0.776536  0.747312   
3          Naive Bayes  0.734375  0.792702   0.758242  0.770950  0.764543   
4        Random Forest  0.784375  0.891874   0.805556  0.810056  0.807799   
5              XGBoost  0.812500  0.878719   0.836158  0.826816  0.831461   

        MCC  
0  0.479299  
1  0.430141  
2  0.399359  
3  0.460015  
4  0.562263  
5  0.620258  


/home/cloud/anaconda3/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [12:59:20] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


app.py created successfully!
